In [10]:
import os
import time
import csv
import threading
import psutil
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.profiler import profile, record_function, ProfilerActivity
from torch.utils.data import Subset

import kagglehub

subset_fraction = 0.005

# 0. Prepare Dataset

In [11]:
kagglehub.login()
"KGAT_40919abd3fa464aee02312875fc62418"

'KGAT_40919abd3fa464aee02312875fc62418'

In [12]:
# Download latest version
path = kagglehub.dataset_download('dimensi0n/imagenet-256')

print("Path to competition files:", path)

Path to competition files: /home/mew/.cache/kagglehub/datasets/dimensi0n/imagenet-256/versions/1


# 1. OS Monitor (ทำงานเบื้องหลัง)

In [13]:
class OSMonitor:
    def __init__(self, filepath, interval=1.0):
        self.filepath = filepath
        self.interval = interval
        self.is_running = False
        self.thread = None

    def _monitor_loop(self):
        with open(self.filepath, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Time_sec', 'CPU_Percent', 'Disk_Read_MB_s', 'Disk_Write_MB_s'])
            start_time = time.time()
            last_disk_io = psutil.disk_io_counters()

            while self.is_running:
                time.sleep(self.interval)
                current_time = time.time() - start_time
                cpu_pct = psutil.cpu_percent(interval=None)
                
                current_disk_io = psutil.disk_io_counters()
                read_mb_s = (current_disk_io.read_bytes - last_disk_io.read_bytes) / (1024 * 1024)
                write_mb_s = (current_disk_io.write_bytes - last_disk_io.write_bytes) / (1024 * 1024)
                last_disk_io = current_disk_io

                writer.writerow([current_time, cpu_pct, read_mb_s, write_mb_s])
                f.flush()

    def start(self):
        self.is_running = True
        self.thread = threading.Thread(target=self._monitor_loop)
        self.thread.start()
        print(f"🔍 เริ่มเก็บข้อมูล OS Metrics: {self.filepath}")

    def stop(self):
        self.is_running = False
        if self.thread:
            self.thread.join()
        print("🛑 หยุดเก็บข้อมูล OS Metrics")

# 2. Setup & Config

In [14]:
class ProfiledImageFolder(datasets.ImageFolder):
    def __getitem__(self, index):
        # 1. จับเวลา: ดึงข้อมูลไฟล์จากฮาร์ดดิสก์ (Disk I/O)
        t_start_io = time.perf_counter()
        path, target = self.samples[index]
        sample = self.loader(path) # จังหวะที่ชี้ไปหาไฟล์ในดิสก์
        disk_time = time.perf_counter() - t_start_io
        
        # 2. จับเวลา: แปลงรูปภาพและทำ Augmentation (CPU Compute)
        t_start_transform = time.perf_counter()
        if self.transform is not None:
            sample = self.transform(sample)
        if self.target_transform is not None:
            target = self.target_transform(target)
        transform_time = time.perf_counter() - t_start_transform
        
        # ส่งค่าเวลากลับไปพร้อมกับรูปภาพด้วย
        return sample, target, disk_time, transform_time

In [15]:
@dataclass
class TrainingConfig:
    data_dir: str = '' # จะถูกเซ็ตออโต้หลังจากรัน kagglehub
    output_dir: str = './thesis_results_real'
    profiler_dir: str = './thesis_results_real/profiler_logs'
    
    batch_size: int = 56  # ระวัง Out of Memory (ปรับลดได้)
    epochs: int = 1        # ทดลองรันแค่ 1 Epoch ก่อนเพื่อให้เห็นผลลัพธ์ไว
    num_workers: int = 2   # ปรับเพื่อทดสอบ I/O Bottleneck
    learning_rate: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 1e-4
    
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

class AverageMeter:
    def __init__(self, name):
        self.name = name
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def accuracy(output, target, topk=(1,)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

# 3. Training Loop (แทรก Profiler + จับเวลา)

In [16]:
def train_epoch(train_loader, model, criterion, optimizer, epoch, config, csv_writer):
    model.train()
    
    # ตัวจับเวลาของ GPU (สร้างไว้หลายๆ ตัวเพื่อแยก Statement การทำงาน)
    events = {
        'h2d_start': torch.cuda.Event(enable_timing=True),
        'h2d_end': torch.cuda.Event(enable_timing=True),
        'fwd_end': torch.cuda.Event(enable_timing=True),
        'bwd_end': torch.cuda.Event(enable_timing=True),
        'opt_end': torch.cuda.Event(enable_timing=True)
    }

    # 🎯 แทนที่ ... ด้วยการตั้งค่า Profiler ของจริง
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
        on_trace_ready=torch.profiler.tensorboard_trace_handler(config.profiler_dir),
        record_shapes=False,
        with_stack=False
    ) as prof: 
        
        end = time.perf_counter()
        
        # รับค่า disk_times และ transform_times มาจาก ProfiledImageFolder
        for i, (images, target, disk_times, transform_times) in enumerate(train_loader):
            
            # เวลาที่ DataLoader รวบรวม Batch เสร็จ (Overhead)
            dataloader_wait_sec = time.perf_counter() - end
            
            # หาค่าเฉลี่ยของเวลาอ่านดิสก์และแปลงรูปใน 1 Batch
            avg_disk_sec = disk_times.float().mean().item()
            avg_transform_sec = transform_times.float().mean().item()

            if config.device == 'cuda':
                events['h2d_start'].record()
            
            # --- 1. เวลาส่งข้อมูลลง GPU (Host to Device) ---
            images = images.to(config.device, non_blocking=True)
            target = target.to(config.device, non_blocking=True)
            
            if config.device == 'cuda':
                events['h2d_end'].record()
            
            # --- 2. เวลา Forward Pass ---
            with record_function("model_forward"):
                outputs = model(images)
                loss = criterion(outputs, target)
            
            if config.device == 'cuda':
                events['fwd_end'].record()
                
            # --- 3. เวลา Backward Pass ---
            with record_function("model_backward"):
                optimizer.zero_grad()
                loss.backward()
            
            if config.device == 'cuda':
                events['bwd_end'].record()
                
            # --- 4. เวลา Optimizer Step ---
            with record_function("optimizer_step"):
                optimizer.step()
            
            if config.device == 'cuda':
                events['opt_end'].record()
                torch.cuda.synchronize() # บังคับรอให้จบเพื่อวัดเวลา
                
                h2d_sec = events['h2d_start'].elapsed_time(events['h2d_end']) / 1000.0
                fwd_sec = events['h2d_end'].elapsed_time(events['fwd_end']) / 1000.0
                bwd_sec = events['fwd_end'].elapsed_time(events['bwd_end']) / 1000.0
                opt_sec = events['bwd_end'].elapsed_time(events['opt_end']) / 1000.0
            else:
                h2d_sec = fwd_sec = bwd_sec = opt_sec = 0.0 

            # บันทึกลง CSV แยกแบบละเอียดยิบ
            csv_writer.writerow([
                epoch, i, 
                avg_disk_sec, avg_transform_sec, dataloader_wait_sec, 
                h2d_sec, fwd_sec, bwd_sec, opt_sec, 
                loss.item()
            ])

            if i % 10 == 0:
                print(f'Batch [{i}]: Disk={avg_disk_sec:.4f}s, Transform={avg_transform_sec:.4f}s, GPU_Fwd={fwd_sec:.4f}s')

            prof.step()
            end = time.perf_counter()

# 4. Main Execution

In [ ]:
def main():
    path = kagglehub.dataset_download('dimensi0n/imagenet-256')
    print(f"✅ โหลดข้อมูลสำเร็จ Path: {path}")

    config = TrainingConfig()
    config.data_dir = path # ชี้ไปที่โฟลเดอร์หลักที่โหลดมา
    
    os.makedirs(config.output_dir, exist_ok=True)
    os.makedirs(config.profiler_dir, exist_ok=True)
    os_csv_path = os.path.join(config.output_dir, 'os_metrics.csv')
    train_csv_path = os.path.join(config.output_dir, 'training_metrics.csv')

    # 🎯 [จุดแก้ไขสำคัญ] ให้ traindir ชี้ไปที่ config.data_dir ตรงๆ ไม่ต้องต่อท้ายด้วย 'train'
    traindir = config.data_dir 
    
    # บรรทัดตรวจสอบซอฟต์แวร์: พิมพ์รายชื่อ 5 โฟลเดอร์แรกออกมาดูเพื่อความชัวร์
    print("📁 ตรวจสอบรายชื่อโฟลเดอร์คลาสย่อย:", os.listdir(traindir)[:5])

    # 2. เริ่มต้น OS Monitor
    monitor = OSMonitor(filepath=os_csv_path, interval=1.0)
    monitor.start()
    
    try:
        # 🎯 จุดแก้ที่ 3 (แนะนำ): ปรับ Resize ให้เข้ากับภาพ 256x256
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        train_dataset = ProfiledImageFolder(
        traindir,
        transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize,
        ]))
        # num_samples = int(len(train_dataset) * subset_fraction)
        # indices = random.sample(range(len(train_dataset)), num_samples)
        # train_dataset_subset = Subset(train_dataset, indices)

        train_loader = DataLoader(
            train_dataset, batch_size=config.batch_size, shuffle=True,
            num_workers=config.num_workers, pin_memory=True, drop_last=True)

        print("กำลังสร้างโมเดล ResNet-50...")
        model = models.mobilenet_v3_small(weights=None).to(config.device)
        criterion = nn.CrossEntropyLoss().to(config.device)
        optimizer = optim.SGD(model.parameters(), config.learning_rate, momentum=config.momentum)

        with open(train_csv_path, 'w', newline='') as csv_file:
            csv_writer = csv.writer(csv_file)
            csv_writer.writerow(['Epoch', 'Batch', 'Data_IO_Sec', 'GPU_Compute_Sec', 'Loss', 'Top1_Acc'])

            for epoch in range(config.epochs):
                print(f"\n--- เริ่มต้น Epoch {epoch} ---")
                train_epoch(train_loader, model, criterion, optimizer, epoch, config, csv_writer)
                csv_file.flush()
                
    finally:
        # ปิด Monitor เสมอไม่ว่าจะเทรนเสร็จหรือเกิด Error
        monitor.stop()

    print(f"\n✅ การทดลองเสร็จสมบูรณ์ ผลลัพธ์อยู่ในโฟลเดอร์ {config.output_dir}")
    print(f"📊 ดู Profiler Trace ด้วยคำสั่ง: tensorboard --logdir={config.profiler_dir}")

In [18]:
main()

✅ โหลดข้อมูลสำเร็จ Path: /home/mew/.cache/kagglehub/datasets/dimensi0n/imagenet-256/versions/1
📁 ตรวจสอบรายชื่อโฟลเดอร์คลาสย่อย: ['tiger', 'tiger_shark', 'gibbon', 'ski_mask', 'bookshop']
🔍 เริ่มเก็บข้อมูล OS Metrics: ./thesis_results_real/os_metrics.csv
กำลังสร้างโมเดล ResNet-50...

--- เริ่มต้น Epoch 0 ---
Batch [0]: Disk=0.0007s, Transform=0.0012s, GPU_Fwd=0.1033s
Batch [10]: Disk=0.0007s, Transform=0.0013s, GPU_Fwd=0.0237s
Batch [20]: Disk=0.0010s, Transform=0.0018s, GPU_Fwd=0.0237s
Batch [30]: Disk=0.0007s, Transform=0.0009s, GPU_Fwd=0.0237s
Batch [40]: Disk=0.0010s, Transform=0.0016s, GPU_Fwd=0.0237s
🛑 หยุดเก็บข้อมูล OS Metrics

✅ การทดลองเสร็จสมบูรณ์ ผลลัพธ์อยู่ในโฟลเดอร์ ./thesis_results_real
📊 ดู Profiler Trace ด้วยคำสั่ง: tensorboard --logdir=./thesis_results_real/profiler_logs
